# M2 · 性能剖析：瓶颈究竟在哪

**核心问题：** 应该优先优化文本编码、GEMM、注意力还是调度开销？分别从 GPU Busy Trace 与端到端关键路径计算，绝不混用分母。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M02_profiling/LESSON.zh-CN.md](../course/M02_profiling/LESSON.zh-CN.md)。切换语言：[English](./02_profile_and_bottlenecks.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
trace = ns.read_json("evidence/results/p2-profile-profile/rocprof-dit/analysis.json")
rows = trace["by_class"]
assert trace["gpu_busy_pct"] == 97.8
ranked = sorted(rows.items(), key=lambda kv: kv[1]["total_s"], reverse=True)
for name, row in ranked:
    print(f"{name:24s} {row['share_of_busy_pct']:5.1f}% GPU-busy / {row['total_s']:.2f}s traced")
plot_values = {name: row["share_of_busy_pct"] for name, row in ranked[:4]}
plot_ylabel, plot_title = "% GPU busy", "Kernel composition (traced; not E2E)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
pipe = ns.read_json("evidence/results/p2-profile-profile/pipeline_breakdown.json")
total = pipe["T_total_warm_s"]
dit = pipe["T_dit_denoise_s"]
share = dit / total
print(f"Warm critical-path DiT: {dit:.3f}/{total:.3f} = {share:.2%}")
print("Cached TE in warm requests:", pipe["T_text_encoder_cached_s"], "s")
assert 0.95 < share < 1.0


## 学生思考 / 自检

列出两个瓶颈占比及其各自分母。为什么 GEMM 占 GPU Busy 61.3%，不能直接说占整个用户请求的 61.3%？

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
